#  C++ Primer : Templates
# 1. One function, many types
* Having written `double average(double a, double b)` you cannot call it with two `long`s and get a `long` back. Writing the function again for every type is tedious, so C++ lets you write it once with the type left as a parameter

```
template<typename T>
T maximum(T a, T b) {
  return (a > b ? a : b);
}
```

* `T` is a type parameter. `maximum(3, 5)` deduces `T = int`, `maximum(3.5, 5.5)` deduces `T = double`
* This is not Python's duck typing happening at run time. The compiler generates _a separate copy of the function for each type you actually use_, compiles each one, and type-checks each one. Nothing is decided while the program runs, and nothing costs anything extra at run time
* The consequences, which are what the exercise is about:
  * an error inside a template is reported only when the template is used with a type that triggers it
  * the error message mentions both the line inside the template and the line that used it, which is why C++ template errors are famously long

# 2. A template parameter can be a number
* A template parameter does not have to be a type. It can be an integer, and then it is an ordinary compile-time constant inside the template

```
template<typename T, long N>
struct point_set {
  long n;
  T x[N];
  T y[N];
};
```

* `point_set<double, 100>` and `point_set<float, 8>` are two different types, each with its arrays the right size, generated from one piece of source
* This is the in-language version of what `-DN=...` did with text substitution in the previous notebook, and it is better in every way that matters:
  * the two sizes can coexist in one program --- with `-D` you get one size per compilation
  * the name is a real constant of a real type, obeying the rules of the language rather than the rules of text substitution; nothing like the `10+10` accident can happen
  * errors are reported in terms of the type, not in terms of mangled text
* The value must be known at compile time. `long n = atol(argv[1]); point_set<double, n> s;` is not possible --- that is what the heap is for
* A struct template's member functions are written inside it as usual and may use `N` freely


In [ ]:
%%writefile tmpl.cc
#include <stdio.h>

template<typename T>
T maximum(T a, T b) {
  return (a > b ? a : b);
}

template<typename T, long N>
struct point_set {
  long n;
  T x[N];
  T y[N];
  long capacity() { return N; }
  T sum_x() {
    T s = 0;
    for (long i = 0; i < n; i++) { s += x[i]; }
    return s;
  }
};

int main() {
  printf("maximum(3, 5)     = %d\n",  maximum(3, 5));
  printf("maximum(3.5, 2.5) = %f\n",  maximum(3.5, 2.5));

  point_set<double, 8>   small;
  point_set<double, 1000> large;

  small.n = 3;
  for (long i = 0; i < small.n; i++) { small.x[i] = i; small.y[i] = 10 * i; }

  printf("small : capacity %ld, sizeof %ld, sum_x %f\n",
         small.capacity(), sizeof(small), small.sum_x());
  printf("large : capacity %ld, sizeof %ld\n",
         large.capacity(), sizeof(large));
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall tmpl.cc -o tmpl

In [ ]:
%%bash
./tmpl

* Two sizes of `point_set` in one program, from one definition, with no run-time cost: `capacity()` returns a constant the compiler already knows

* <font color="green">Exercise: (a) try `maximum(3, 5.5)` and read the error --- why can the compiler not deduce `T`? Fix it by writing `maximum<double>(3, 5.5)`. (b) add a `point_set<float, 8>` and compare its `sizeof` with `point_set<double, 8>`. (c) try to make the size a run-time value (`long m = 8; point_set<double, m> s;`) and read the error. (d) call `maximum(p, q)` with two `vec`s from the struct notebook --- where is the error reported, and what does it say is missing?</font>

* Start from this copy


In [ ]:
%%writefile tmpl_ex.cc
#include <stdio.h>

struct vec {
  double x;
  double y;
};

template<typename T>
T maximum(T a, T b) {
  return (a > b ? a : b);
}

template<typename T, long N>
struct point_set {
  long n;
  T x[N];
  T y[N];
  long capacity() { return N; }
};

int main() {
  /* (a) */
  /* printf("%f\n", maximum(3, 5.5)); */

  /* (b) */
  point_set<double, 8> d8;
  printf("sizeof(point_set<double, 8>) = %ld\n", sizeof(d8));

  /* (c) long m = 8; point_set<double, m> s; */

  /* (d) vec p = { 3, 4 }, q = { 1, 2 }; maximum(p, q); */
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall tmpl_ex.cc -o tmpl_ex

In [ ]:
%%bash
./tmpl_ex